# Finger Pose Classifier

This notebook aims to use traditional classifiers to determine finger curl. Finger curl has been segmented into 5 classes:

| Class             | MCP      | PIP      | Thumb      |
|-------------------|----------|----------|------------|
| curll_curl        | curl     | curl     | ~straight  |
| curl_straight     | curl     | straight | ~straight  |
| straight_curl     | straight | curl     | ~straight  |
| straight_straight | straight | straight | ~straight  |
| thumb_curl        | straight | straight | curl       |

\
There is a separate thumb curl class because even when making a fist the the flex sensors and IMU's don't observe a substantial displacement, visually they don't seem to move too much. This thumb curl class has the thumb fully curled, with the tip of the thumb reaching into the centre of the palm.

## 1 Importing Libraries

In [1]:
# Run once to install any missing packages
import subprocess, sys
pkgs = ['scikit-learn', 'pandas', 'numpy', 'matplotlib', 'seaborn', 'scipy']
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs)
print('Dependencies ready.')


Dependencies ready.


In [2]:
import os, glob, re, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import signal as scipy_signal
from scipy.stats import skew, kurtosis

from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score
)

# Classifiers
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
print('Imports OK.')

Imports OK.


## 2 Parameter Configuration

In [3]:
# =============================================================================
# Path to the root folder containing gesture subfolders (ThesisA.data).
# Each subfolder name becomes the class label.
# Accepts absolute paths or paths relative to this notebook.
DATA_ROOT = '../ML/NewTrainingData/Finger_MCP_PIP'   # <-- change this to point to your data folder

# Labels to INCLUDE. Set to None to use ALL subfolders automatically.
# Example: INCLUDE_LABELS = ['TwoHand_L_Fist_R_Fist', 'TwoHand_L_Flat_R_Flat']
INCLUDE_LABELS = ["curl_curl", "curl_straight", "straight_curl", "straight_straight"]

# =============================================================================
# 1B. COLUMN SELECTION
# =============================================================================
# Choose which sensor modalities to include.
# Each flag independently includes/excludes that group of columns.

USE_YAW_PITCH_ROLL   = True    # IMU Euler angles (yaw, pitch, roll / heading)
USE_QUATERNIONS      = False   # Raw quaternion components (quat_w/x/y/z)
                               # Note: for TwoHand_L_Flat_R_Flat, quaternions are
                               #       the primary data — pitch/roll/yaw are derived.
                               #       Set True if using that label.
USE_ACCELEROMETER    = True    # Linear accelerometer (ax, ay, az)
USE_FLEX_SENSORS     = True    # Flex sensor readings (mcp_flex, pip_flex)

# Choose which body segments to include.
# Available: 'palm_prox', 'thumb', 'index', 'middle', 'ring', 'pinky', 'wrist'
# 'palm_mid'  = excluded — IMU columns exist in CSV but contain no sensor data
# 'palm_prox' = IMU on back of palm (proximal, near wrist)
# 'wrist'     = IMU at end of forearm closest to wrist
# Note: flex sensors exist for fingers only (thumb, index, middle, ring, pinky) — NOT palm
INCLUDE_SEGMENTS = [
    'palm_mid',  
    # 'palm_prox',#  excluded: IMU columns exist but contain no data
    'thumb',
    'index',
    'middle',
    'ring',
    'pinky',
    'wrist',
]

# Choose which hands to include
INCLUDE_HANDS = ['left', 'right']  # options: 'left', 'right', or both
FINGERS = ["thumb", "index", "middle", "ring", "pinky"]

# =============================================================================
# 1C. PREPROCESSING
# =============================================================================

# --- Resampling ---
# Resample each trial to a fixed number of time steps (handles variable-length files).
# Set to None to skip resampling (all files must then have the same row count).
RESAMPLE_TO_N_STEPS = 90       # target number of rows per trial

# --- Low-pass Butterworth filter ---
APPLY_BUTTERWORTH    = True     # Apply low-pass filter to smooth IMU noise
BUTTERWORTH_CUTOFF   = 5.0      # Cutoff frequency in Hz
BUTTERWORTH_ORDER    = 4        # Filter order
SAMPLING_RATE_HZ     = 30.0     # Approximate sampling rate of the glove

# --- Normalisation ---
# 'standard'  → zero mean, unit variance (StandardScaler) — good for SVM, LDA
# 'minmax'    → scale to [0, 1]  (MinMaxScaler) — good for KNN, neural nets
# None        → no normalisation
NORMALISATION = 'standard'

# --- Feature extraction mode ---
# 'flatten'    → use the raw resampled time series as a flat feature vector
#                (n_steps × n_channels features per sample)
# 'stats'      → extract statistical features per channel
#                (mean, std, min, max, range, RMS, skew, kurtosis)
# 'fft'        → FFT magnitude spectrum per channel
# 'stats+fft'  → combine statistical + FFT features
FEATURE_MODE = 'stats'    # Recommended starting point

# =============================================================================
# 1D. TRAIN / TEST SPLIT
# =============================================================================
TEST_SIZE        = 0.2    # Fraction of data held out for testing
RANDOM_STATE     = 42
CV_FOLDS         = 5      # Number of cross-validation folds

# =============================================================================
# 1E. ALGORITHMS
# =============================================================================
# Set True/False to include/exclude each algorithm.
ALGORITHMS = {
    'SVM (RBF)'           : True,
    'SVM (Linear)'        : True,
    'Random Forest'       : True,
    'KNN'                 : True,
    'Gradient Boosting'   : True,   # Slower — enable when needed
    'Logistic Regression' : True,
    'LDA'                 : True,
}

print('Configuration loaded.')
print(f'  Data root  : {DATA_ROOT}')
print(f'  Hands      : {INCLUDE_HANDS}')
print(f'  Segments   : {INCLUDE_SEGMENTS}')
print(f'  Modalities : YPR={USE_YAW_PITCH_ROLL}, Quat={USE_QUATERNIONS}, Accel={USE_ACCELEROMETER}, Flex={USE_FLEX_SENSORS}')
print(f'  Features   : {FEATURE_MODE}')
print(f'  Algorithms : {[k for k,v in ALGORITHMS.items() if v]}')

Configuration loaded.
  Data root  : ../ML/NewTrainingData/Finger_MCP_PIP
  Hands      : ['left', 'right']
  Segments   : ['palm_mid', 'thumb', 'index', 'middle', 'ring', 'pinky', 'wrist']
  Modalities : YPR=True, Quat=False, Accel=True, Flex=True
  Features   : stats
  Algorithms : ['SVM (RBF)', 'SVM (Linear)', 'Random Forest', 'KNN', 'Gradient Boosting', 'Logistic Regression', 'LDA']


## 3 Data Loading

In [4]:
# ── Build column selector from config ─────────────────────────────────────────

# All 295 columns, grouped by type for easy selection
META_COLS = [
    'run_index', 'request_id', 'request_ts',
    'right_recv_time_ms', 'right_glove_time_ms', 'right_time',
    'left_recv_time_ms',  'left_glove_time_ms',  'left_time',
    'left_hand', 'right_hand'
]

SEGMENTS_WITH_FLEX = ['thumb', 'index', 'middle', 'ring', 'pinky']  # palm has no flex sensors
IMU_LOCATIONS = ['mid', 'prox']  # distal phalanx / proximal phalanx

def build_sensor_columns(hands, segments, use_ypr, use_quat, use_accel, use_flex):
    """Return list of sensor column names matching the user config."""
    cols = []
    for hand in hands:
        for seg in segments:
            # Determine IMU sub-locations for this segment
            # palm has mid + prox; all finger segments have mid + prox; wrist has no sub-location
            if seg == 'wrist':
                prefix = f'{hand}_wrist'
                if use_ypr:
                    cols += [f'{prefix}_heading', f'{prefix}_pitch', f'{prefix}_roll']
                if use_quat:
                    cols += [f'{prefix}_quat_w', f'{prefix}_quat_x',
                             f'{prefix}_quat_y', f'{prefix}_quat_z']
                if use_accel:
                    cols += [f'{prefix}_ax', f'{prefix}_ay', f'{prefix}_az']
                # wrist has no flex sensor
            else:
                # segment has mid + prox IMU positions
                # For 'thumb','index' etc, map to e.g. 'left_thumb_mid_yaw'
                for loc in ['mid', 'prox']:
                    prefix = f'{hand}_{seg}_{loc}'
                    if use_ypr:
                        cols += [f'{prefix}_yaw', f'{prefix}_pitch', f'{prefix}_roll']
                    if use_quat:
                        cols += [f'{prefix}_quat_w', f'{prefix}_quat_x',
                                 f'{prefix}_quat_y', f'{prefix}_quat_z']
                    if use_accel:
                        cols += [f'{prefix}_ax', f'{prefix}_ay', f'{prefix}_az']
                # Flex sensors (one per finger joint combination, not per loc)
                if use_flex and seg in ['thumb', 'index', 'middle', 'ring', 'pinky']:  # palm has no flex
                    cols += [f'{hand}_{seg}_mcp_flex', f'{hand}_{seg}_pip_flex']
    return cols

# Map user-friendly segment names to CSV naming convention
seg_map = {
    'palm_mid':  'palm',   # excluded by default — no sensor data on palm_mid
    # 'palm_prox': 'palm',   # deduplicated below
    'thumb':     'thumb',
    'index':     'index',
    'middle':    'middle',
    'ring':      'ring',
    'pinky':     'pinky',
    'wrist':     'wrist',
}

# Resolve segments, de-duplicating palm if both palm_mid and palm_prox selected
resolved_segs = list(dict.fromkeys([seg_map[s] for s in INCLUDE_SEGMENTS]))

SENSOR_COLS = build_sensor_columns(
    hands    = INCLUDE_HANDS,
    segments = resolved_segs,
    use_ypr  = USE_YAW_PITCH_ROLL,
    use_quat = USE_QUATERNIONS,
    use_accel= USE_ACCELEROMETER,
    use_flex = USE_FLEX_SENSORS
)

print(f'Selected {len(SENSOR_COLS)} sensor columns.')
print('First 10:', SENSOR_COLS[:10])
print('Last 10:', SENSOR_COLS[-10:])

Selected 176 sensor columns.
First 10: ['left_palm_mid_yaw', 'left_palm_mid_pitch', 'left_palm_mid_roll', 'left_palm_mid_ax', 'left_palm_mid_ay', 'left_palm_mid_az', 'left_palm_prox_yaw', 'left_palm_prox_pitch', 'left_palm_prox_roll', 'left_palm_prox_ax']
Last 10: ['right_pinky_prox_ay', 'right_pinky_prox_az', 'right_pinky_mcp_flex', 'right_pinky_pip_flex', 'right_wrist_heading', 'right_wrist_pitch', 'right_wrist_roll', 'right_wrist_ax', 'right_wrist_ay', 'right_wrist_az']


In [5]:
# ── Load all CSV files ─────────────────────────────────────────────────────────

def load_dataset(data_root, include_labels, sensor_cols):
    """Load all CSVs from label subfolders. Returns list of (df_trial, label)."""
    data_root = os.path.expanduser(data_root)
    if not os.path.isdir(data_root):
        raise FileNotFoundError(
            f"DATA_ROOT not found: '{data_root}'\n"
            "Please update DATA_ROOT in the Configuration cell to point to your "
            "ThesisA.data folder."
        )

    label_dirs = sorted([
        d for d in os.listdir(data_root)
        if os.path.isdir(os.path.join(data_root, d))
        and d not in ('PDF', 'sdb', 'idk')  # skip non-gesture folders
    ])

    if include_labels is not None:
        label_dirs = [d for d in label_dirs if d in include_labels]

    if not label_dirs:
        raise ValueError('No label folders found. Check DATA_ROOT and INCLUDE_LABELS.')

    print(f'Found {len(label_dirs)} gesture classes:')
    trials, labels = [], []

    for label in label_dirs:
        folder = os.path.join(data_root, label)
        csv_files = sorted(glob.glob(os.path.join(folder, '*.csv')))
        print(f'  [{label}]  →  {len(csv_files)} files')
        for fpath in csv_files:
            try:
                df = pd.read_csv(fpath)
                # Keep only sensor columns that exist in this file
                available = [c for c in sensor_cols if c in df.columns]
                if not available:
                    print(f'    WARNING: no matching sensor columns in {os.path.basename(fpath)}')
                    continue
                trials.append(df[available].values.astype(np.float32))
                labels.append(label)
            except Exception as e:
                print(f'    ERROR loading {os.path.basename(fpath)}: {e}')

    print(f'\nTotal trials loaded: {len(trials)}')
    return trials, labels, label_dirs

trials_raw, labels_raw, class_names = load_dataset(
    DATA_ROOT, INCLUDE_LABELS, SENSOR_COLS
)

# Class distribution
from collections import Counter
print('\nClass distribution:')
for cls, cnt in sorted(Counter(labels_raw).items()):
    print(f'  {cls}: {cnt} trials')

Found 4 gesture classes:
  [curl_curl]  →  20 files
  [curl_straight]  →  20 files
  [straight_curl]  →  20 files
  [straight_straight]  →  20 files

Total trials loaded: 80

Class distribution:
  curl_curl: 20 trials
  curl_straight: 20 trials
  straight_curl: 20 trials
  straight_straight: 20 trials


In [6]:
# print(trials_raw[0].shape)  # shape of first trial (n_steps, n_channels)

## 4. Preprocessing

In [7]:
# ── Step 1: Resample to fixed length ──────────────────────────────────────────

def resample_trial(trial, n_steps):
    """Resample a (T, C) array to (n_steps, C) using linear interpolation."""
    T, C = trial.shape
    if T == n_steps:
        return trial
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, n_steps)
    resampled = np.zeros((n_steps, C), dtype=np.float32)
    for c in range(C):
        resampled[:, c] = np.interp(new_idx, old_idx, trial[:, c])
    return resampled

if RESAMPLE_TO_N_STEPS is not None:
    trials_resampled = [resample_trial(t, RESAMPLE_TO_N_STEPS) for t in trials_raw]
    print(f'Resampled all trials to {RESAMPLE_TO_N_STEPS} time steps.')
else:
    trials_resampled = trials_raw
    lengths = [t.shape[0] for t in trials_resampled]
    print(f'No resampling. Trial lengths: min={min(lengths)}, max={max(lengths)}, mean={np.mean(lengths):.1f}')
    if min(lengths) != max(lengths):
        print('  WARNING: Variable-length trials detected. Enable RESAMPLE_TO_N_STEPS '
              'or use a feature extraction mode that handles variable length.')

print(f'Trial shape: {trials_resampled[0].shape}  (time_steps × channels)')

Resampled all trials to 90 time steps.
Trial shape: (90, 176)  (time_steps × channels)


In [8]:
# ── Step 2: Butterworth low-pass filter ───────────────────────────────────────

def apply_butterworth(trials, cutoff, order, fs):
    """Apply a zero-phase Butterworth low-pass filter to each channel of each trial."""
    nyq = fs / 2.0
    norm_cutoff = cutoff / nyq
    if norm_cutoff >= 1.0:
        print(f'  WARNING: cutoff {cutoff} Hz >= Nyquist {nyq} Hz — skipping filter.')
        return trials
    b, a = scipy_signal.butter(order, norm_cutoff, btype='low', analog=False)
    filtered = []
    for t in trials:
        t_filt = scipy_signal.filtfilt(b, a, t, axis=0).astype(np.float32)
        filtered.append(t_filt)
    return filtered

if APPLY_BUTTERWORTH:
    trials_filtered = apply_butterworth(
        trials_resampled, BUTTERWORTH_CUTOFF, BUTTERWORTH_ORDER, SAMPLING_RATE_HZ
    )
    print(f'Applied Butterworth low-pass filter: {BUTTERWORTH_CUTOFF} Hz cutoff, '
          f'order {BUTTERWORTH_ORDER}, Nyquist = {SAMPLING_RATE_HZ/2} Hz')
else:
    trials_filtered = trials_resampled
    print('Butterworth filter skipped.')

Applied Butterworth low-pass filter: 5.0 Hz cutoff, order 4, Nyquist = 15.0 Hz


The data has been resampled to be of size 90 rows x 176 sensor columns, and a butterworth filter has been applied to all the sensor columns to smooth the readings.

Now we need to split up each of the trials by finger. We need a dataset for each finger on each hand.

In [9]:
# ── Split processed trials into 10 finger datasets ─────────────────────────────

FINGERS = ['thumb', 'index', 'middle', 'ring', 'pinky']
HANDS = ['left', 'right']

def get_finger_columns(sensor_cols, hand, finger, palm_segment='palm_mid'):
    """
    For one hand/finger pair, return:
      - that finger's columns
      - that hand's palm columns
    using the already-selected SENSOR_COLS ordering.
    """
    finger_prefix = f"{hand}_{finger}_"
    palm_prefix = f"{hand}_{palm_segment}_"

    finger_cols = [c for c in sensor_cols if c.startswith(finger_prefix)]
    palm_cols = [c for c in sensor_cols if c.startswith(palm_prefix)]

    return palm_cols + finger_cols

def split_trials_by_finger(trials, labels, sensor_cols, hands=None, fingers=None, palm_segment='palm_mid'):
    """
    Split list of processed trial arrays into 10 datasets:
      (left/right) x (thumb/index/middle/ring/pinky)

    Returns
    -------
    finger_data : dict
        key   = (hand, finger)
        value = {
            'raw': list of np.ndarray with shape (T, C_finger),
            'labels': list of class labels aligned with raw,
            'columns': ordered column names used for that dataset
        }
    """
    hands = HANDS if hands is None else hands
    fingers = FINGERS if fingers is None else fingers

    col_index = {c: i for i, c in enumerate(sensor_cols)}
    finger_data = {}

    for hand in hands:
        for finger in fingers:
            key = (hand, finger)
            cols = get_finger_columns(sensor_cols, hand, finger, palm_segment=palm_segment)
            idxs = [col_index[c] for c in cols if c in col_index]

            finger_data[key] = {
                "raw": [trial[:, idxs] for trial in trials],
                "labels": list(labels),
                "columns": cols,
            }

    return finger_data

finger_trials = split_trials_by_finger(
    trials_filtered,
    labels_raw,
    SENSOR_COLS,
    hands=INCLUDE_HANDS,
    fingers=FINGERS,
    palm_segment='palm_mid'   # change to 'palm_prox' if that is what you really want
)

print("Built per-finger datasets:")
for key in finger_trials:
    first_shape = finger_trials[key]['raw'][0].shape if len(finger_trials[key]['raw']) else None
    print(f"{key}: {len(finger_trials[key]['raw'])} trials, shape per trial = {first_shape}, channels = {len(finger_trials[key]['columns'])}")

# Inspect columns for right thumb and right ring
print("\nRight Thumb columns:")
print(finger_trials[('right', 'thumb')]['columns'])
print("\nRight Ring columns:")
print(finger_trials[('right', 'ring')]['columns'])

Built per-finger datasets:
('left', 'thumb'): 80 trials, shape per trial = (90, 20), channels = 20
('left', 'index'): 80 trials, shape per trial = (90, 20), channels = 20
('left', 'middle'): 80 trials, shape per trial = (90, 20), channels = 20
('left', 'ring'): 80 trials, shape per trial = (90, 20), channels = 20
('left', 'pinky'): 80 trials, shape per trial = (90, 20), channels = 20
('right', 'thumb'): 80 trials, shape per trial = (90, 20), channels = 20
('right', 'index'): 80 trials, shape per trial = (90, 20), channels = 20
('right', 'middle'): 80 trials, shape per trial = (90, 20), channels = 20
('right', 'ring'): 80 trials, shape per trial = (90, 20), channels = 20
('right', 'pinky'): 80 trials, shape per trial = (90, 20), channels = 20

Right Thumb columns:
['right_palm_mid_yaw', 'right_palm_mid_pitch', 'right_palm_mid_roll', 'right_palm_mid_ax', 'right_palm_mid_ay', 'right_palm_mid_az', 'right_thumb_mid_yaw', 'right_thumb_mid_pitch', 'right_thumb_mid_roll', 'right_thumb_mid_ax',

In [10]:
# ── Data augmentation utilities for time-series trials ─────────────────────────

def add_gaussian_noise(trial, noise_std=0.01):
    """
    trial: (T, C) np.ndarray
    Adds zero-mean Gaussian noise with given std (relative to channel scale).
    """
    # scale noise per channel based on its std, to keep it proportional
    channel_std = np.std(trial, axis=0, keepdims=True) + 1e-8
    noise = np.random.normal(loc=0.0, scale=noise_std, size=trial.shape) * channel_std
    return trial + noise.astype(np.float32)

def random_time_warp(trial, max_scale=0.1):
    """
    Randomly stretch/compress time axis by up to ±max_scale and resample to original length.
    This preserves (T, C) shape.
    """
    T, C = trial.shape
    # sample scale factor in [1-max_scale, 1+max_scale]
    scale = 1.0 + np.random.uniform(-max_scale, max_scale)
    new_T = int(round(T * scale))
    old_idx = np.linspace(0, 1, T)
    new_idx = np.linspace(0, 1, new_T)

    warped = np.zeros((new_T, C), dtype=np.float32)
    for c in range(C):
        warped[:, c] = np.interp(new_idx, old_idx, trial[:, c])

    # resample back to length T
    return resample_trial(warped, T)  # reuses your existing resample_trial()[file:191]

def random_channel_scaling(trial, scale_range=(0.9, 1.1)):
    """
    Randomly scale each channel by a factor in scale_range.
    """
    T, C = trial.shape
    scales = np.random.uniform(scale_range[0], scale_range[1], size=(1, C))
    return (trial * scales).astype(np.float32)

In [11]:
# ── Build augmented copies of each finger dataset ──────────────────────────────

def augment_finger_data(
        finger_data,
        n_aug_per_trial=1,
        use_noise=True,
        use_time_warp=True,
        use_scaling=False):
    """
    For each (hand, finger), create augmented trials and labels.

    n_aug_per_trial: how many augmented versions to create per original trial.
    Returns a NEW dict with original + augmented data mixed together.
    """
    aug_finger_data = {}

    for key, fd in finger_data.items():
        raw_trials = fd["raw"]
        labels = fd["labels"]
        cols = fd["columns"]

        new_raw = list(raw_trials)      # start with originals
        new_labels = list(labels)

        for trial, label in zip(raw_trials, labels):
            for _ in range(n_aug_per_trial):
                t_aug = trial.copy()

                if use_time_warp:
                    t_aug = random_time_warp(t_aug, max_scale=0.1)

                if use_noise:
                    t_aug = add_gaussian_noise(t_aug, noise_std=0.02)

                if use_scaling:
                    t_aug = random_channel_scaling(t_aug, scale_range=(0.9, 1.1))

                new_raw.append(t_aug)
                new_labels.append(label)

        aug_finger_data[key] = {
            "raw": new_raw,
            "labels": new_labels,
            "columns": cols,
        }

    return aug_finger_data


def augment_raw_trials(
        raw_trials,
        labels,
        n_aug_per_trial=1,
        use_noise=True,
        use_time_warp=True,
        use_scaling=False):
    """
    Augment a set of raw trials and labels.
    Returns augmented trials and corresponding labels.
    """
    new_raw = list(raw_trials)
    new_labels = list(labels)

    for trial, label in zip(raw_trials, labels):
        for _ in range(n_aug_per_trial):
            t_aug = trial.copy()

            if use_time_warp:
                t_aug = random_time_warp(t_aug, max_scale=0.1)

            if use_noise:
                t_aug = add_gaussian_noise(t_aug, noise_std=0.02)

            if use_scaling:
                t_aug = random_channel_scaling(t_aug, scale_range=(0.9, 1.1))

            new_raw.append(t_aug)
            new_labels.append(label)

    return new_raw, new_labels

# `finger_data_aug` is created later after raw train/test splitting to avoid leakage.


mid         = 3x accel + 3x ypr             = 6\
prox        = 3x accel + 3x ypr             = 6\
palm_mid    = 3x accel + 3x ypr             = 6\
flex        = mcp + pip                     = 2\
\
total       = mid + prox + palm_mid + flex  = 20\

## 5. Feature Extraction

In [19]:
# ── Step 3: Feature extraction ────────────────────────────────────────────────

def extract_stats(trial):
    """Statistical features per channel: mean, std, min, max, range, RMS, skew, kurtosis."""
    feats = np.concatenate([
        trial.mean(axis=0),
        trial.std(axis=0),
        trial.min(axis=0),
        trial.max(axis=0),
        trial.max(axis=0) - trial.min(axis=0),          # range
        np.sqrt((trial**2).mean(axis=0)),                # RMS
        skew(trial, axis=0).astype(np.float32),
        kurtosis(trial, axis=0).astype(np.float32),
    ])
    return feats


def extract_fft(trial, fs):
    """FFT magnitude features per channel (first half of spectrum)."""
    n = trial.shape[0]
    fft_mag = np.abs(np.fft.rfft(trial, axis=0))  # shape: (n//2+1, C)
    return fft_mag.flatten().astype(np.float32)


def extract_features(trials, mode, fs):
    X = []
    for t in trials:
        if mode == 'flatten':
            feats = t.flatten()
        elif mode == 'stats':
            feats = extract_stats(t)
        elif mode == 'fft':
            feats = extract_fft(t, fs)
        elif mode == 'stats+fft':
            feats = np.concatenate([extract_stats(t), extract_fft(t, fs)])
        else:
            raise ValueError(f'Unknown FEATURE_MODE: {mode}')
        X.append(feats)
    return np.array(X, dtype=np.float32)


# print(type(finger_trials))

finger_data_aug = {}

for key in finger_trials:
    raw_trials = finger_trials[key]['raw']
    labels = finger_trials[key]['labels']
    cols = finger_trials[key]['columns']

    raw_train, raw_test, labels_train, labels_test = train_test_split(
        raw_trials,
        labels,
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=labels,
    )

    raw_train_aug, labels_train_aug = augment_raw_trials(
        raw_train,
        labels_train,
        n_aug_per_trial=3,
        use_noise=True,
        use_time_warp=True,
        use_scaling=False,
    )

    print(f"=== Dataset: {key} ===")
    print(f"  Original samples: {len(raw_trials)}")
    print(f"  Train originals: {len(raw_train)}")
    print(f"  Train augmented: {len(raw_train_aug)}")
    print(f"  Test samples: {len(raw_test)}")

    le = LabelEncoder()
    le.fit(labels)

    X_train = extract_features(raw_train_aug, FEATURE_MODE, SAMPLING_RATE_HZ)
    X_test = extract_features(raw_test, FEATURE_MODE, SAMPLING_RATE_HZ)

    for dataset_name, X in [('train', X_train), ('test', X_test)]:
        bad = np.isnan(X) | np.isinf(X)
        if bad.any():
            print(f'WARNING: {bad.sum()} NaN/Inf values found in {dataset_name} set — replacing with 0.')
        if dataset_name == 'train':
            X_train = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
        else:
            X_test = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print(f'Feature matrix shapes: train={X_train.shape}, test={X_test.shape}')

    y_train = le.transform(labels_train_aug)
    y_test = le.transform(labels_test)

    finger_data_aug[key] = {
        'raw_train': raw_train_aug,
        'raw_test': raw_test,
        'labels_train': labels_train_aug,
        'labels_test': labels_test,
        'columns': cols,
        'X_train': X_train,
        'X_test': X_test,
        'y_train': y_train,
        'y_test': y_test,
        'le': le,
    }


=== Dataset: ('left', 'thumb') ===
  Original samples: 80
  Train originals: 64
  Train augmented: 256
  Test samples: 16
Feature matrix shapes: train=(256, 160), test=(16, 160)
=== Dataset: ('left', 'index') ===
  Original samples: 80
  Train originals: 64
  Train augmented: 256
  Test samples: 16
Feature matrix shapes: train=(256, 160), test=(16, 160)
=== Dataset: ('left', 'middle') ===
  Original samples: 80
  Train originals: 64
  Train augmented: 256
  Test samples: 16
Feature matrix shapes: train=(256, 160), test=(16, 160)
=== Dataset: ('left', 'ring') ===
  Original samples: 80
  Train originals: 64
  Train augmented: 256
  Test samples: 16
Feature matrix shapes: train=(256, 160), test=(16, 160)
=== Dataset: ('left', 'pinky') ===
  Original samples: 80
  Train originals: 64
  Train augmented: 256
  Test samples: 16
Feature matrix shapes: train=(256, 160), test=(16, 160)
=== Dataset: ('right', 'thumb') ===
  Original samples: 80
  Train originals: 64
  Train augmented: 256
  Test

We've split the data by finger, we've extracted statistical and fft features from each channel. As result, we've flattened the data into a 1D dataset of 160 features.

## 6. Train | Test Split

In [20]:
for key in finger_data_aug:
    print(f"\n=== Dataset: {key} ===")
    # print(f"{key} X shape:", finger_data_aug[key]['X'].shape)
    # print(f"{key} y shape:", finger_data_aug[key]['y'].shape)
    # print(f"{key} Classes:", list(finger_data_aug[key]['le'].classes_))

    # X_train, X_test, y_train, y_test = train_test_split(
    # finger_data_aug[key]['X'], finger_data_aug[key]['y'], test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=finger_data_aug[key]['y']
    # )

    if NORMALISATION == 'standard':
        scaler = StandardScaler()
    elif NORMALISATION == 'minmax':
        scaler = MinMaxScaler()
    else:
        scaler = None

    if scaler is not None:
        X_train = scaler.fit_transform(finger_data_aug[key]['X_train'])
        X_test  = scaler.transform(finger_data_aug[key]['X_test'])
        print(f'Applied {NORMALISATION} normalisation.')
    else:
        print('No normalisation applied.')

    finger_data_aug[key].update({'X_train': X_train, 'X_test': X_test, 'y_train': y_train, 'y_test': y_test})

    print(f'Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')


=== Dataset: ('left', 'thumb') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('left', 'index') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('left', 'middle') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('left', 'ring') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('left', 'pinky') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('right', 'thumb') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('right', 'index') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('right', 'middle') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('right', 'ring') ===
Applied standard normalisation.
Train: 256 samples | Test: 16 samples

=== Dataset: ('right', 'pinky') ===
Appli

In [21]:
# ── Define classifiers from config ────────────────────────────────────────────

CLASSIFIER_DEFS = {
    'SVM (RBF)':           SVC(kernel='rbf', C=10, gamma='scale', random_state=RANDOM_STATE),
    'SVM (Linear)':        SVC(kernel='linear', C=1, random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'KNN':                 KNeighborsClassifier(n_neighbors=5, metric='euclidean'),
    'Gradient Boosting':   GradientBoostingClassifier(n_estimators=100, random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
    'LDA':                 LinearDiscriminantAnalysis(),
}

active_classifiers = {k: v for k, v in CLASSIFIER_DEFS.items() if ALGORITHMS.get(k, False)}
print(f'Running {len(active_classifiers)} algorithms: {list(active_classifiers.keys())}')

Running 7 algorithms: ['SVM (RBF)', 'SVM (Linear)', 'Random Forest', 'KNN', 'Gradient Boosting', 'Logistic Regression', 'LDA']


In [22]:
# ── Print model parameters for all active classifiers ─────────────────────────
# Shows every hyperparameter and (after training) learned model properties.

PARAM_DESCRIPTIONS = {
    # SVM
    'C':               'Regularisation — higher = fits training data more tightly, lower = wider margin (less overfit)',
    'kernel':          'Decision boundary shape: rbf=curved (non-linear), linear=flat hyperplane',
    'gamma':           'RBF kernel width — scale=1/(n_features*X.var()), auto=1/n_features. Higher=more complex boundary',
    # Random Forest
    'n_estimators':    'Number of decision trees in the forest — more trees = more stable, slower',
    'max_depth':       'Max tree depth — None=grow until pure leaves (may overfit)',
    'min_samples_split':'Min samples required to split a node — higher=simpler trees',
    'min_samples_leaf': 'Min samples required at a leaf — higher=smoother decision boundary',
    'max_features':    'Features considered per split — sqrt=sqrt(n_features), None=all features',
    # KNN
    'n_neighbors':     'Number of nearest neighbours to vote — lower=more sensitive to noise, higher=smoother boundary',
    'metric':          'Distance measure used to find neighbours',
    'weights':         'uniform=all neighbours equal vote, distance=closer neighbours weighted more',
    # Logistic Regression
    'max_iter':        'Maximum solver iterations — increase if convergence warning appears',
    'C':               'Inverse regularisation strength — higher=less regularisation (fits training harder)',
    'solver':          'Optimisation algorithm used to fit the model',
    'multi_class':     'Strategy for multi-class: auto selects ovr or multinomial',
    # LDA
    'solver':          'svd=no matrix inversion (stable), lsqr/eigen=faster but less stable',
    'shrinkage':       'Regularisation on covariance matrix — None=no shrinkage',
    'n_components':    'Dimensions to reduce to — None=min(n_classes-1, n_features)',
    # Gradient Boosting
    'learning_rate':   'Shrinks each tree contribution — lower=more trees needed but better generalisation',
    'subsample':       'Fraction of samples per tree — <1.0 adds randomness (stochastic boosting)',
}

print('=' * 70)
print('CLASSIFIER PARAMETERS')
print('=' * 70)

for name, clf in active_classifiers.items():
    params = clf.get_params()
    print(f'\n┌─ {name} ─────────────────────────────────────────')
    print(f'│  Type: {type(clf).__name__}')
    print(f'│')
    for param, value in params.items():
        desc = PARAM_DESCRIPTIONS.get(param, '')
        desc_str = f'  # {desc}' if desc else ''
        print(f'│  {param:<22} = {str(value):<15}{desc_str}')
    print(f'└──────────────────────────────────────────────────')

# ── Post-training learned properties (run after Section 4 training cell) ─────
print()
print('=' * 70)
print('POST-TRAINING LEARNED PROPERTIES  (run after Section 4)')
print('=' * 70)

if 'results' not in dir():
    print('  Classifiers not yet trained — run Section 4 first.')
else:
    for name, res in results.items():
        clf = res['clf']
        print(f'\n  {name}:')

        if hasattr(clf, 'support_vectors_'):
            n_sv = clf.support_vectors_.shape[0]
            pct  = 100 * n_sv / len(X_train)
            print(f'    Support vectors : {n_sv} / {len(X_train)} training samples ({pct:.1f}%)')
            print(f'    (Low % = confident, wide margin. High % = decision boundary uncertain)')

        if hasattr(clf, 'n_iter_'):
            print(f'    Iterations      : {clf.n_iter_}')

        if hasattr(clf, 'feature_importances_'):
            fi   = clf.feature_importances_
            top3 = fi.argsort()[::-1][:3]
            print(f'    Feature importances (top 3 feature indices): {top3}')
            print(f'    Top 3 importance values: {fi[top3].round(4)}')

        if hasattr(clf, 'coef_'):
            coef = clf.coef_
            print(f'    Coefficient matrix shape: {coef.shape}  ({coef.shape[0]} classes × {coef.shape[1]} features)')
            print(f'    Coef magnitude  : mean={abs(coef).mean():.4f}, max={abs(coef).max():.4f}')

        if hasattr(clf, 'scalings_'):
            print(f'    LDA scalings shape: {clf.scalings_.shape}  ({clf.scalings_.shape[1]} discriminant axes)')
            print(f'    Explained variance ratio: {clf.explained_variance_ratio_.round(4) if hasattr(clf, "explained_variance_ratio_") else "N/A"}')
        if hasattr(clf, 'estimators_') and name == 'Random Forest':
            depths = [e.get_depth() for e in clf.estimators_]
            print(f'    Trees grown     : {len(clf.estimators_)}')
            print(f'    Tree depth      : mean={sum(depths)/len(depths):.1f}, max={max(depths)}, min={min(depths)}')


CLASSIFIER PARAMETERS

┌─ SVM (RBF) ─────────────────────────────────────────
│  Type: SVC
│
│  C                      = 10               # Inverse regularisation strength — higher=less regularisation (fits training harder)
│  break_ties             = False          
│  cache_size             = 200            
│  class_weight           = None           
│  coef0                  = 0.0            
│  decision_function_shape = ovr            
│  degree                 = 3              
│  gamma                  = scale            # RBF kernel width — scale=1/(n_features*X.var()), auto=1/n_features. Higher=more complex boundary
│  kernel                 = rbf              # Decision boundary shape: rbf=curved (non-linear), linear=flat hyperplane
│  max_iter               = -1               # Maximum solver iterations — increase if convergence warning appears
│  probability            = False          
│  random_state           = 42             
│  shrinking              = True           


In [23]:
finger_classifiers = {}
accuracies = {}

for key in finger_data_aug:
    # print(f"\n=== Training classifiers for dataset: {key} ===")
    X_train = finger_data_aug[key]['X_train']
    y_train = finger_data_aug[key]['y_train']
    X_test  = finger_data_aug[key]['X_test']
    y_test  = finger_data_aug[key]['y_test']

    results = {}
    for name, clf in active_classifiers.items():
        # print(f'\nTraining {name}...')
        clf.fit(X_train, y_train)
        y_pred = clf.predict(X_test)
        acc = accuracy_score(y_test, y_pred)
        # print(f'  Test Accuracy: {acc:.4f}')
        results[name] = {'clf': clf, 'accuracy': acc}
        accuracies.setdefault(key, {})[name] = acc

    finger_classifiers[key] = results

# Display accuracies in tabular form

df = pd.DataFrame(accuracies).T
print("\nAccuracies Table:")
print(df.to_string())


Accuracies Table:
              SVM (RBF)  SVM (Linear)  Random Forest     KNN  Gradient Boosting  Logistic Regression     LDA
left  thumb      0.4375        0.5000         0.5625  0.5000             0.5000               0.3750  0.4375
      index      0.8125        0.8750         0.9375  0.8125             0.8750               0.8750  0.6875
      middle     0.6875        0.6250         0.8750  0.3750             0.8750               0.6250  0.6875
      ring       0.8125        0.8750         0.8750  0.5000             0.5625               0.8125  0.7500
      pinky      0.6875        0.8125         0.8750  0.5625             0.6250               0.8125  0.8750
right thumb      0.8750        0.8750         0.9375  0.7500             0.8125               0.8750  0.9375
      index      0.6875        0.8750         0.9375  0.5000             0.9375               0.9375  0.9375
      middle     0.5625        0.6875         0.9375  0.5000             0.6875               0.6250  0.6875
